# Logistic Regression

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler

# --- Load ---
df = pd.read_csv("/Users/dmk6603/Documents/ransom_victims/5-swdb_prob_wrong_right/data/swdb_tech.csv")  # adjust filename

# --- Explode technologies into binary columns ---
tech_series = df["technologies"].str.split("; ")
all_techs = tech_series.explode()
tech_counts = all_techs.value_counts()
keep_techs = tech_counts[tech_counts >= 200].index.tolist()
print(f"Technologies kept (≥200): {len(keep_techs)}")

tech_dummies = tech_series.apply(lambda x: pd.Series({t: 1 for t in x if t in keep_techs})).fillna(0).astype(int)

# --- Features ---
X = tech_dummies.copy()
X["log_revenue"] = np.log1p(df["site_revenue_usd"])
X["log_employees"] = np.log1p(df["site_employees"])
# X["sector"] = df["sector"]  # uncomment if you have it
# sector_dummies = pd.get_dummies(X["sector"], prefix="sec", drop_first=True)
# X = pd.concat([X.drop("sector", axis=1), sector_dummies], axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
feature_names = X.columns.tolist()

# --- LASSO per group (≥50 victims) ---
group_counts = df["group_name"].value_counts()
groups = group_counts[group_counts >= 50].index.tolist()
print(f"Groups to test: {len(groups)}")

results = []
for g in groups:
    y = (df["group_name"] == g).astype(int)
    model = LogisticRegressionCV(
        penalty="l1", solver="saga", Cs=20,
        cv=5, scoring="roc_auc", max_iter=5000, random_state=42
    )
    model.fit(X_scaled, y)
    coefs = pd.Series(model.coef_[0], index=feature_names)
    nonzero = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    
    print(f"\n{'='*60}")
    print(f"GROUP: {g} (n={y.sum()}, AUC={model.scores_[1].mean(axis=0).max():.3f})")
    print(f"Selected features: {len(nonzero)}")
    print(nonzero.head(15).to_string())
    
    for feat, coef in nonzero.items():
        results.append({"group": g, "feature": feat, "coef": coef})

# --- Export ---
results_df = pd.DataFrame(results)
results_df.to_csv("lasso_results.csv", index=False)
print(f"\nSaved to lasso_results.csv ({len(results_df)} rows)")

Technologies kept (≥200): 313
Groups to test: 20

GROUP: lockbit3 (n=411, AUC=0.519)
Selected features: 11
Microsoft SQL Server Reporting Services   -0.094305
G Suite                                    0.080922
Microsoft Exchange 2003                   -0.053871
Tableau Public                             0.049852
American Express                          -0.047923
Adobe Illustrator                         -0.021423
Apple iPad                                -0.020384
Microsoft TFS                             -0.015186
OpenResty                                  0.011848
AT&T                                       0.002187
AppNexus                                  -0.000269

GROUP: play (n=379, AUC=0.592)
Selected features: 263
Dell PowerEdge T100 Tower Server    0.637950
IBM Softlayer DNS                   0.427314
Dell PowerEdge M620 Blade Server    0.313569
Dell PowerEdge T110 Tower Server   -0.312511
Zoom                               -0.311572
Amazon AWS Elastic Beanstalk       -0.305

/opt/homebrew/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(



GROUP: akira (n=174, AUC=0.544)
Selected features: 5
Amazon EC2     -0.118807
Pardot         -0.030850
cPanel          0.030107
Apple iPhone   -0.013926
SMARTPHONE     -0.013926

GROUP: qilin (n=167, AUC=0.605)
Selected features: 59
Microsoft Outlook                      0.209814
React                                 -0.208141
Microsoft System Center               -0.158572
AngularJS                             -0.148319
Bootstrap v320                         0.111088
ArcGIS                                 0.097783
Hotjar                                 0.097741
GitHub                                -0.096502
Windows XP                             0.092739
Platfora                              -0.090315
Google Analytics Enhanced Ecommerce    0.088861
Apache Tomcat                         -0.086208
Vimeo                                  0.085034
Amazon Simple Email Service (SES)     -0.081630
Portable PC                           -0.081519

GROUP: blackbasta (n=154, AUC=0.643)
Selected